In [36]:
from networks import GAT, PPGAT
import torch
from torch.utils.data import random_split
from sklearn.model_selection import train_test_split

from torch_geometric.data import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool
import torch.nn.functional as F
from torch.nn import Linear
import torch.nn as nn
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np
import random
from sklearn.model_selection import StratifiedKFold



In [37]:
#set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

In [38]:
 #load datasets
target = 'Q16790'
dataset = torch.load(f'datasets/Gdatasets/{target}_dataset.pt', weights_only=False)
rg_dataset = torch.load(f'datasets/RGdatasets/{target}_RG_dataset.pt', weights_only=False)
ppgat_dataset = torch.load(f'datasets/PPGATdatasets/{target}_PPGAT_dataset.pt', weights_only=False)

### Ablation study for protein class Q16790

NO node features case

In [39]:
# Remove node features
for data in dataset:
    num_nodes = data.num_nodes
    data.x = torch.zeros((num_nodes, 1))   # 1-dimensional dummy feature

for data in rg_dataset:
    num_nodes = data.num_nodes
    data.x = torch.zeros((num_nodes, 1))   # 1-dimensional dummy feature

for data in ppgat_dataset:
    num_nodes = data.num_nodes
    data.x = torch.zeros((num_nodes, 1))   # 1-dimensional dummy feature

In [40]:
#  using the same train/val/test split for all three models ito fairly compare them 

# Create labels for stratification (optional but good)
labels = [int(data.y.item()) for data in dataset]

smiles_list = [data.smiles for data in dataset]


# First: train_val vs. test split
train_val_smiles, test_smiles = train_test_split(
    smiles_list,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# Then: train vs. val split (from train_val)
train_smiles, val_smiles = train_test_split(
    train_val_smiles,
    test_size=0.125,  # ~10% of full data
    stratify=[labels[smiles_list.index(s)] for s in train_val_smiles],
    random_state=42
)

def subset_by_smiles(dataset, smiles_subset):
    smiles_set = set(smiles_subset)
    return [data for data in dataset if data.smiles in smiles_set]

train_set     = subset_by_smiles(dataset, train_smiles)
val_set       = subset_by_smiles(dataset, val_smiles)
test_set      = subset_by_smiles(dataset, test_smiles)

train_rg      = subset_by_smiles(rg_dataset, train_smiles)
val_rg        = subset_by_smiles(rg_dataset, val_smiles)
test_rg       = subset_by_smiles(rg_dataset, test_smiles)

train_ppgat    = subset_by_smiles(ppgat_dataset, train_smiles)
val_ppgat      = subset_by_smiles(ppgat_dataset, val_smiles)
test_ppgat     = subset_by_smiles(ppgat_dataset, test_smiles)

In [41]:
def evaluate_model(model, dataloader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for data in dataloader:
            data = data.to(device)
            out = model(data)
            probs = torch.sigmoid(out).view(-1).cpu().numpy()
            labels = data.y.view(-1).cpu().numpy()

            # Ensure probs and labels are arrays, even for batch size 1
            probs = np.atleast_1d(probs)
            labels = np.atleast_1d(labels)

            all_probs.extend(probs)
            all_labels.extend(labels)

    y_pred = (np.array(all_probs) > 0.5).astype(int)
    acc = accuracy_score(all_labels, y_pred)
    auroc = roc_auc_score(all_labels, all_probs)
    return acc, auroc

In [42]:
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=32)
test_loader  = DataLoader(test_set, batch_size=32)

/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


# GAT

In [44]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
criterion = nn.BCEWithLogitsLoss()

k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
train_y = np.array([d.y.item() for d in train_set])


accs, aucs = [], []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_y)), train_y)):
    print(f"\nFOLD {fold+1}")

    # fold-specific train/val splits inside the training set
    fold_train_set = [train_set[i] for i in train_idx]
    fold_val_set   = [train_set[i] for i in val_idx]

    train_loader = DataLoader(fold_train_set, batch_size=32, shuffle=True)
    val_loader   = DataLoader(fold_val_set, batch_size=32)
    
    # always the same external test set
    test_loader = DataLoader(test_set, batch_size=32)

    # ---- recreate model ----
    model = GAT(
        in_channels=1,
        edge_attr_dim=dataset[0].edge_attr.size(1),
        hidden_channels=64,
        out_channels=1,
        heads=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    #  train 
    for epoch in range(1, 101):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()
            out = model(batch).view(-1)
            y_batch = batch.y.float().view(-1)

            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 5 == 0:
            print(f"Epoch {epoch:02d} | Loss = {total_loss/len(train_loader):.4f}")

    # evaluate on test set 
    acc, auroc = evaluate_model(model, test_loader, device)

    accs.append(acc)
    aucs.append(auroc)

    print(f"[Fold {fold+1}] ACC = {acc:.4f} | AUROC = {auroc:.4f}")


FOLD 1


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6940
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6933
Epoch 20 | Loss = 0.6933
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 1] ACC = 0.5000 | AUROC = 0.5000

FOLD 2


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6933
Epoch 10 | Loss = 0.6934
Epoch 15 | Loss = 0.6934
Epoch 20 | Loss = 0.6933
Epoch 25 | Loss = 0.6933
Epoch 30 | Loss = 0.6941
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 2] ACC = 0.5000 | AUROC = 0.5000

FOLD 3


...

Epoch 05 | Loss = 0.6933
Epoch 10 | Loss = 0.6932
Epoch 15 | Loss = 0.6932
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 3] ACC = 0.5000 | AUROC = 0.5000

FOLD 4


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6940
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6933
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6934
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 4] ACC = 0.5000 | AUROC = 0.5000

FOLD 5


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6939
Epoch 10 | Loss = 0.6936
Epoch 15 | Loss = 0.6935
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6933
Epoch 30 | Loss = 0.6933
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 5] ACC = 0.5000 | AUROC = 0.5000


In [45]:
print("\n GAT FINAL CROSS-VALIDATION RESULTS")
print(f"Mean ACC:   {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Mean AUROC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")


 GAT FINAL CROSS-VALIDATION RESULTS
Mean ACC:   0.5000 ± 0.0000
Mean AUROC: 0.5000 ± 0.0000


# GAT-rg

In [46]:
train_loader = DataLoader(train_rg, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_rg, batch_size=32)
test_loader  = DataLoader(test_rg, batch_size=32)

/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [47]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
criterion = nn.BCEWithLogitsLoss()

k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
train_y = np.array([d.y.item() for d in train_rg])


accs, aucs = [], []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_y)), train_y)):
    print(f"\nFOLD {fold+1}")

    # fold-specific train/val splits inside the training set
    fold_train_set = [train_rg[i] for i in train_idx]
    fold_val_set   = [train_rg[i] for i in val_idx]

    train_loader = DataLoader(fold_train_set, batch_size=32, shuffle=True)
    val_loader   = DataLoader(fold_val_set, batch_size=32)
    
    # always the same external test set
    test_loader = DataLoader(test_rg, batch_size=32)

    # ---- recreate model ----
    model = GAT(
        in_channels=1,
        edge_attr_dim=dataset[0].edge_attr.size(1),
        hidden_channels=64,
        out_channels=1,
        heads=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    # ---- train ----
    for epoch in range(1, 101):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()
            out = model(batch).view(-1)
            y_batch = batch.y.float().view(-1)

            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 5 == 0:
            print(f"Epoch {epoch:02d} | Loss = {total_loss/len(train_loader):.4f}")

    # ---- evaluate on test set ----
    acc, auroc = evaluate_model(model, test_loader, device)

    accs.append(acc)
    aucs.append(auroc)

    print(f"[Fold {fold+1}] ACC = {acc:.4f} | AUROC = {auroc:.4f}")


FOLD 1


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6935
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6932
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 1] ACC = 0.5000 | AUROC = 0.5007

FOLD 2


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6935
Epoch 10 | Loss = 0.6944
Epoch 15 | Loss = 0.6933
Epoch 20 | Loss = 0.6933
Epoch 25 | Loss = 0.6933
Epoch 30 | Loss = 0.6933
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6935
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 2] ACC = 0.5000 | AUROC = 0.5007

FOLD 3


...

Epoch 05 | Loss = 0.6939
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6935
Epoch 20 | Loss = 0.6933
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 3] ACC = 0.5000 | AUROC = 0.4993

FOLD 4


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6941
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6934
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 4] ACC = 0.5000 | AUROC = 0.5007

FOLD 5


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6939
Epoch 10 | Loss = 0.6936
Epoch 15 | Loss = 0.6936
Epoch 20 | Loss = 0.6933
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 5] ACC = 0.5000 | AUROC = 0.5007


In [48]:
print("\n GAT-rg FINAL CROSS-VALIDATION RESULTS")
print(f"Mean ACC:   {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Mean AUROC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")


 GAT-rg FINAL CROSS-VALIDATION RESULTS
Mean ACC:   0.5000 ± 0.0000
Mean AUROC: 0.5004 ± 0.0005


## PPGAT

In [49]:
train_loader = DataLoader(train_ppgat, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ppgat, batch_size=32)
test_loader  = DataLoader(test_ppgat, batch_size=32)

/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [50]:

accs, aucs = [], []
device = 'cuda' if torch.cuda.is_available() else 'cpu'
criterion = nn.BCEWithLogitsLoss()

k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
train_y = np.array([d.y.item() for d in train_ppgat])


for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_y)), train_y)):
    print(f"\nFOLD {fold+1}")

    # fold-specific train/val splits inside the training set
    fold_train_set = [train_ppgat[i] for i in train_idx]
    fold_val_set   = [train_ppgat[i] for i in val_idx]

    train_loader = DataLoader(fold_train_set, batch_size=32, shuffle=True)
    val_loader   = DataLoader(fold_val_set, batch_size=32)
    
    # always the same external test set
    test_loader = DataLoader(test_ppgat, batch_size=32)

    #  model
    model = PPGAT(
        in_channels=1,
        edge_attr_dim=dataset[0].edge_attr.size(1),
        hidden_channels=64,
        out_channels=1,
        heads=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    # ---- train ----
    for epoch in range(1, 101):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()
            out = model(batch).view(-1)
            y_batch = batch.y.float().view(-1)

            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 5 == 0:
            print(f"Epoch {epoch:02d} | Loss = {total_loss/len(train_loader):.4f}")

    # evaluate on test set 
    acc, auroc = evaluate_model(model, test_loader, device)

    accs.append(acc)
    aucs.append(auroc)

    print(f"[Fold {fold+1}] ACC = {acc:.4f} | AUROC = {auroc:.4f}")


FOLD 1


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6950
Epoch 10 | Loss = 0.6936
Epoch 15 | Loss = 0.6933
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6937
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 1] ACC = 0.5000 | AUROC = 0.5000

FOLD 2


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6944
Epoch 10 | Loss = 0.6937
Epoch 15 | Loss = 0.6936
Epoch 20 | Loss = 0.6940
Epoch 25 | Loss = 0.6935
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6933
Epoch 40 | Loss = 0.6933
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6935
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 2] ACC = 0.5000 | AUROC = 0.5000

FOLD 3


...

Epoch 05 | Loss = 0.6943
Epoch 10 | Loss = 0.6932
Epoch 15 | Loss = 0.6933
Epoch 20 | Loss = 0.6929
Epoch 25 | Loss = 0.6933
Epoch 30 | Loss = 0.6935
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6934
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 3] ACC = 0.5000 | AUROC = 0.5000

FOLD 4


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6936
Epoch 10 | Loss = 0.6932
Epoch 15 | Loss = 0.6931
Epoch 20 | Loss = 0.6939
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6931
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 4] ACC = 0.5000 | AUROC = 0.5000

FOLD 5


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6937
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6936
Epoch 20 | Loss = 0.6933
Epoch 25 | Loss = 0.6934
Epoch 30 | Loss = 0.6933
Epoch 35 | Loss = 0.6933
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 5] ACC = 0.5000 | AUROC = 0.5000


In [61]:
print("\n PPGAT FINAL CROSS-VALIDATION RESULTS")
print(f"Mean ACC:   {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Mean AUROC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")


 PPGAT FINAL CROSS-VALIDATION RESULTS
Mean ACC:   0.9206 ± 0.0051
Mean AUROC: 0.9732 ± 0.0014


### CASE no edges

In [52]:
#load datasets
target = 'Q16790'
dataset = torch.load(f'datasets/Gdatasets/{target}_dataset.pt', weights_only=False)
rg_dataset = torch.load(f'datasets/RGdatasets/{target}_RG_dataset.pt', weights_only=False)
rgnn_dataset = torch.load(f'datasets/PPGATdatasets/{target}_PPGAT_dataset.pt', weights_only=False)

In [53]:
# Remove edges 
for data in dataset:
    num_nodes = data.num_nodes
    data.edge_index = torch.empty((2, 0), dtype=torch.long)  # no edges
    data.edge_attr  = torch.empty((0, data.edge_attr.size(1))) if data.edge_attr is not None else None

for data in rg_dataset:
    num_nodes = data.num_nodes
    data.x = data.x  # keep node features

    # Remove all edges
    data.edge_index = torch.empty((2, 0), dtype=torch.long)

    # Force edge_attr to have shape (0, 6) always
    data.edge_attr = torch.empty((0, 6), dtype=torch.float)
 

for data in rgnn_dataset:
    num_nodes = data.num_nodes
    data.edge_index = torch.empty((2, 0), dtype=torch.long)
    data.edge_attr  = torch.empty((0, data.edge_attr.size(1))) if data.edge_attr is not None else None

In [54]:
# Create labels for stratification (optional but good)
labels = [int(data.y.item()) for data in dataset]

smiles_list = [data.smiles for data in dataset]


# First: train_val vs. test split
train_val_smiles, test_smiles = train_test_split(
    smiles_list,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# Then: train vs. val split (from train_val)
train_smiles, val_smiles = train_test_split(
    train_val_smiles,
    test_size=0.125,  # ~10% of full data
    stratify=[labels[smiles_list.index(s)] for s in train_val_smiles],
    random_state=42
)

def subset_by_smiles(dataset, smiles_subset):
    smiles_set = set(smiles_subset)
    return [data for data in dataset if data.smiles in smiles_set]

train_set     = subset_by_smiles(dataset, train_smiles)
val_set       = subset_by_smiles(dataset, val_smiles)
test_set      = subset_by_smiles(dataset, test_smiles)

train_rg      = subset_by_smiles(rg_dataset, train_smiles)
val_rg        = subset_by_smiles(rg_dataset, val_smiles)
test_rg       = subset_by_smiles(rg_dataset, test_smiles)

train_rgnn    = subset_by_smiles(rgnn_dataset, train_smiles)
val_rgnn      = subset_by_smiles(rgnn_dataset, val_smiles)
test_rgnn     = subset_by_smiles(rgnn_dataset, test_smiles)

### GAT

In [55]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
criterion = nn.BCEWithLogitsLoss()
k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
train_y = np.array([d.y.item() for d in train_set])


accs, aucs = [], []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_y)), train_y)):
    print(f"\nFOLD {fold+1}")

    # fold-specific train/val splits inside the training set
    fold_train_set = [train_set[i] for i in train_idx]
    fold_val_set   = [train_set[i] for i in val_idx]

    train_loader = DataLoader(fold_train_set, batch_size=32, shuffle=True)
    val_loader   = DataLoader(fold_val_set, batch_size=32)
    
    # always same test set
    test_loader = DataLoader(test_set, batch_size=32)

    # model-
    model = GAT(
        in_channels=19,
        edge_attr_dim=dataset[0].edge_attr.size(1),
        hidden_channels=64,
        out_channels=1,
        heads=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    #  train 
    for epoch in range(1, 101):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()
            out = model(batch).view(-1)
            y_batch = batch.y.float().view(-1)

            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 5 == 0:
            print(f"Epoch {epoch:02d} | Loss = {total_loss/len(train_loader):.4f}")

    # evaluate on test set 
    acc, auroc = evaluate_model(model, test_loader, device)

    accs.append(acc)
    aucs.append(auroc)

    print(f"[Fold {fold+1}] ACC = {acc:.4f} | AUROC = {auroc:.4f}")


FOLD 1


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6933
Epoch 10 | Loss = 0.6932
Epoch 15 | Loss = 0.6932
Epoch 20 | Loss = 0.6933
Epoch 25 | Loss = 0.6933
Epoch 30 | Loss = 0.6933
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6931
Epoch 80 | Loss = 0.6933
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 1] ACC = 0.5000 | AUROC = 0.5000

FOLD 2


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6934
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6932
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6933
Epoch 30 | Loss = 0.6933
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6934
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6933
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6933
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6933
Epoch 100 | Loss = 0.6932
[Fold 2] ACC = 0.5000 | AUROC = 0.5000

FOLD 3


...

Epoch 05 | Loss = 0.6933
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6932
Epoch 20 | Loss = 0.6934
Epoch 25 | Loss = 0.6933
Epoch 30 | Loss = 0.6933
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6933
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6934
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6933
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6933
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 3] ACC = 0.5000 | AUROC = 0.5000

FOLD 4


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6933
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6934
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6933
Epoch 35 | Loss = 0.6933
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6935
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6933
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 4] ACC = 0.5000 | AUROC = 0.5000

FOLD 5


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6933
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6933
Epoch 20 | Loss = 0.6933
Epoch 25 | Loss = 0.6933
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6937
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6933
Epoch 50 | Loss = 0.6934
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6933
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6933
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 5] ACC = 0.5000 | AUROC = 0.5000


In [56]:
print("\n GAT FINAL CROSS-VALIDATION RESULTS - No edges")
print(f"Mean ACC:   {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Mean AUROC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")


 GAT FINAL CROSS-VALIDATION RESULTS - No edges
Mean ACC:   0.5000 ± 0.0000
Mean AUROC: 0.5000 ± 0.0000


### GAT-rg

In [57]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
criterion = nn.BCEWithLogitsLoss()
k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
train_y = np.array([d.y.item() for d in train_rg])


accs, aucs = [], []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_y)), train_y)):
    print(f"\nFOLD {fold+1}")

    # fold-specific train/val splits inside the training set
    fold_train_set = [train_rg[i] for i in train_idx]
    fold_val_set   = [train_rg[i] for i in val_idx]

    train_loader = DataLoader(fold_train_set, batch_size=32, shuffle=True)
    val_loader   = DataLoader(fold_val_set, batch_size=32)
    
    test_loader = DataLoader(test_rg, batch_size=32)

    model = GAT(
        in_channels=19,
        edge_attr_dim=dataset[0].edge_attr.size(1),
        hidden_channels=64,
        out_channels=1,
        heads=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    #train
    for epoch in range(1, 101):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()
            out = model(batch).view(-1)
            y_batch = batch.y.float().view(-1)

            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 5 == 0:
            print(f"Epoch {epoch:02d} | Loss = {total_loss/len(train_loader):.4f}")

    # evaluate on test set
    acc, auroc = evaluate_model(model, test_loader, device)

    accs.append(acc)
    aucs.append(auroc)

    print(f"[Fold {fold+1}] ACC = {acc:.4f} | AUROC = {auroc:.4f}")


FOLD 1


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6932
Epoch 10 | Loss = 0.6932
Epoch 15 | Loss = 0.6933
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6933
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6934
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6933
Epoch 80 | Loss = 0.6933
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 1] ACC = 0.5000 | AUROC = 0.5000

FOLD 2


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6933
Epoch 10 | Loss = 0.6932
Epoch 15 | Loss = 0.6933
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6933
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6933
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6933
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6933
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 2] ACC = 0.5000 | AUROC = 0.5000

FOLD 3


...

Epoch 05 | Loss = 0.6932
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6932
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6933
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 3] ACC = 0.5000 | AUROC = 0.5000

FOLD 4


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6934
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6933
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6932
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6932
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 4] ACC = 0.5000 | AUROC = 0.5000

FOLD 5


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.6933
Epoch 10 | Loss = 0.6933
Epoch 15 | Loss = 0.6932
Epoch 20 | Loss = 0.6932
Epoch 25 | Loss = 0.6933
Epoch 30 | Loss = 0.6932
Epoch 35 | Loss = 0.6933
Epoch 40 | Loss = 0.6932
Epoch 45 | Loss = 0.6932
Epoch 50 | Loss = 0.6932
Epoch 55 | Loss = 0.6932
Epoch 60 | Loss = 0.6932
Epoch 65 | Loss = 0.6932
Epoch 70 | Loss = 0.6932
Epoch 75 | Loss = 0.6932
Epoch 80 | Loss = 0.6932
Epoch 85 | Loss = 0.6932
Epoch 90 | Loss = 0.6932
Epoch 95 | Loss = 0.6932
Epoch 100 | Loss = 0.6932
[Fold 5] ACC = 0.5000 | AUROC = 0.5000


In [58]:
print("\n GAT-rg FINAL CROSS-VALIDATION RESULTS - No edges")
print(f"Mean ACC:   {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Mean AUROC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")


 GAT-rg FINAL CROSS-VALIDATION RESULTS - No edges
Mean ACC:   0.5000 ± 0.0000
Mean AUROC: 0.5000 ± 0.0000


#### PPGAT

In [59]:

accs, aucs = [], []
device = 'cuda' if torch.cuda.is_available() else 'cpu'
criterion = nn.BCEWithLogitsLoss()

k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
train_y = np.array([d.y.item() for d in train_rgnn])


for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_y)), train_y)):
    print(f"\nFOLD {fold+1}")

    # fold-specific train/val splits inside the training set
    fold_train_set = [train_rgnn[i] for i in train_idx]
    fold_val_set   = [train_rgnn[i] for i in val_idx]

    train_loader = DataLoader(fold_train_set, batch_size=32, shuffle=True)
    val_loader   = DataLoader(fold_val_set, batch_size=32)
    
    test_loader = DataLoader(test_rgnn, batch_size=32)

   
    model = PPGAT(
        in_channels=19,
        edge_attr_dim=dataset[0].edge_attr.size(1),
        hidden_channels=64,
        out_channels=1,
        heads=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    # train 
    for epoch in range(1, 101):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()
            out = model(batch).view(-1)
            y_batch = batch.y.float().view(-1)

            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 5 == 0:
            print(f"Epoch {epoch:02d} | Loss = {total_loss/len(train_loader):.4f}")

    #evaluate on test set
    acc, auroc = evaluate_model(model, test_loader, device)

    accs.append(acc)
    aucs.append(auroc)

    print(f"[Fold {fold+1}] ACC = {acc:.4f} | AUROC = {auroc:.4f}")


FOLD 1


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.3597
Epoch 10 | Loss = 0.2970
Epoch 15 | Loss = 0.2682
Epoch 20 | Loss = 0.2506
Epoch 25 | Loss = 0.2349
Epoch 30 | Loss = 0.2274
Epoch 35 | Loss = 0.2195
Epoch 40 | Loss = 0.2232
Epoch 45 | Loss = 0.2108
Epoch 50 | Loss = 0.2051
Epoch 55 | Loss = 0.1965
Epoch 60 | Loss = 0.1914
Epoch 65 | Loss = 0.1825
Epoch 70 | Loss = 0.1851
Epoch 75 | Loss = 0.1790
Epoch 80 | Loss = 0.1740
Epoch 85 | Loss = 0.1649
Epoch 90 | Loss = 0.1646
Epoch 95 | Loss = 0.1512
Epoch 100 | Loss = 0.1580
[Fold 1] ACC = 0.9233 | AUROC = 0.9741

FOLD 2


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.3461
Epoch 10 | Loss = 0.2863
Epoch 15 | Loss = 0.2731
Epoch 20 | Loss = 0.2530
Epoch 25 | Loss = 0.2479
Epoch 30 | Loss = 0.2364
Epoch 35 | Loss = 0.2337
Epoch 40 | Loss = 0.2186
Epoch 45 | Loss = 0.2121
Epoch 50 | Loss = 0.2147
Epoch 55 | Loss = 0.2032
Epoch 60 | Loss = 0.2038
Epoch 65 | Loss = 0.1894
Epoch 70 | Loss = 0.1863
Epoch 75 | Loss = 0.1845
Epoch 80 | Loss = 0.1775
Epoch 85 | Loss = 0.1762
Epoch 90 | Loss = 0.1672
Epoch 95 | Loss = 0.1669
Epoch 100 | Loss = 0.1604
[Fold 2] ACC = 0.9266 | AUROC = 0.9738

FOLD 3


...

Epoch 05 | Loss = 0.3612
Epoch 10 | Loss = 0.2884
Epoch 15 | Loss = 0.2609
Epoch 20 | Loss = 0.2482
Epoch 25 | Loss = 0.2412
Epoch 30 | Loss = 0.2246
Epoch 35 | Loss = 0.2184
Epoch 40 | Loss = 0.2097
Epoch 45 | Loss = 0.2050
Epoch 50 | Loss = 0.2003
Epoch 55 | Loss = 0.1987
Epoch 60 | Loss = 0.1909
Epoch 65 | Loss = 0.1836
Epoch 70 | Loss = 0.1830
Epoch 75 | Loss = 0.1798
Epoch 80 | Loss = 0.1803
Epoch 85 | Loss = 0.1773
Epoch 90 | Loss = 0.1652
Epoch 95 | Loss = 0.1783
Epoch 100 | Loss = 0.1618
[Fold 3] ACC = 0.9213 | AUROC = 0.9747

FOLD 4


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.3795
Epoch 10 | Loss = 0.2996
Epoch 15 | Loss = 0.2810
Epoch 20 | Loss = 0.2654
Epoch 25 | Loss = 0.2605
Epoch 30 | Loss = 0.2448
Epoch 35 | Loss = 0.2387
Epoch 40 | Loss = 0.2334
Epoch 45 | Loss = 0.2279
Epoch 50 | Loss = 0.2169
Epoch 55 | Loss = 0.2149
Epoch 60 | Loss = 0.2056
Epoch 65 | Loss = 0.1990
Epoch 70 | Loss = 0.1971
Epoch 75 | Loss = 0.2007
Epoch 80 | Loss = 0.1930
Epoch 85 | Loss = 0.1889
Epoch 90 | Loss = 0.1809
Epoch 95 | Loss = 0.1751
Epoch 100 | Loss = 0.1754
[Fold 4] ACC = 0.9114 | AUROC = 0.9709

FOLD 5


/home/tejaurrutoam1/.conda/envs/pyg_v3/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 05 | Loss = 0.3443
Epoch 10 | Loss = 0.3022
Epoch 15 | Loss = 0.2753
Epoch 20 | Loss = 0.2553
Epoch 25 | Loss = 0.2532
Epoch 30 | Loss = 0.2359
Epoch 35 | Loss = 0.2247
Epoch 40 | Loss = 0.2209
Epoch 45 | Loss = 0.2076
Epoch 50 | Loss = 0.2005
Epoch 55 | Loss = 0.2057
Epoch 60 | Loss = 0.1955
Epoch 65 | Loss = 0.1835
Epoch 70 | Loss = 0.1851
Epoch 75 | Loss = 0.1811
Epoch 80 | Loss = 0.1692
Epoch 85 | Loss = 0.1763
Epoch 90 | Loss = 0.1704
Epoch 95 | Loss = 0.1611
Epoch 100 | Loss = 0.1594
[Fold 5] ACC = 0.9206 | AUROC = 0.9727


In [60]:
print("\n RGNN FINAL CROSS-VALIDATION RESULTS - No edges")
print(f"Mean ACC:   {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Mean AUROC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")


 RGNN FINAL CROSS-VALIDATION RESULTS - No edges
Mean ACC:   0.9206 ± 0.0051
Mean AUROC: 0.9732 ± 0.0014
